In [11]:
# MERGING AND STANDARDISING SOCIAL MEDIA DATASETS INTO A SINGLE ONE
# Imports
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import rasterio
import numpy as np
import rasterio
from scipy.spatial import cKDTree


# --- Configuration ---
work_dir = r"C:\Users\karasoo1\Downloads"

# Input filenames
flic_file = os.path.join(work_dir, "flic.shp")
twitt_file = os.path.join(work_dir, "twitt.shp")
vk_file = os.path.join(work_dir, "vk_east.shp")

# Output filenames
out_4326 = os.path.join(work_dir, "social_merged_4326.gpkg")
out_3857 = os.path.join(work_dir, "social_merged_3857.gpkg")

def clean_and_merge():
    print("1. Loading and standardizing 'flic' data...")
    gdf_flic = gpd.read_file(flic_file)
    
    # Drop existing 'fid' to avoid conflicts
    if 'fid' in gdf_flic.columns:
        gdf_flic = gdf_flic.drop(columns=['fid'])
    
    # Standardize columns
    gdf_flic['layer'] = 'flic'
    if gdf_flic.crs is None:
        gdf_flic.set_crs(epsg=4326, inplace=True)
    else:
        gdf_flic = gdf_flic.to_crs(epsg=4326)


    print("2. Loading and standardizing 'twitt' data...")
    gdf_twitt = gpd.read_file(twitt_file)
    
    if 'fid' in gdf_twitt.columns:
        gdf_twitt = gdf_twitt.drop(columns=['fid'])
        
    gdf_twitt['layer'] = 'twitt'
    if gdf_twitt.crs is None:
        gdf_twitt.set_crs(epsg=4326, inplace=True)
    else:
        gdf_twitt = gdf_twitt.to_crs(epsg=4326)


    print("3. Loading and cleaning 'vk_east' data...")
    # --- FIX IS HERE: Added encoding='cp1251' ---
    try:
        gdf_vk = gpd.read_file(vk_file, encoding='cp1251')
    except Exception as e:
        print(f"   Failed with cp1251, trying fallback. Error: {e}")
        # Fallback if cp1251 fails (rare for VK, but possible)
        gdf_vk = gpd.read_file(vk_file, encoding='latin1') 
    
    # A. Rename columns
    rename_map = {
        'photo_id': 'id',
        'owner_id': 'user_id',
        'lat': 'latitude',
        'long': 'longitude',
        'date': 'datetaken_raw' 
    }
    gdf_vk = gdf_vk.rename(columns=rename_map)
    
    # B. Fix Date Format
    print("   Parsing VK dates...")
    gdf_vk['datetaken'] = pd.to_datetime(gdf_vk['datetaken_raw'], errors='coerce').dt.strftime('%Y-%m-%d')
    
    # C. Add Layer Name
    gdf_vk['layer'] = 'vk'
    
    # D. Handle CRS 
    if gdf_vk.crs is None:
        gdf_vk.set_crs(epsg=4326, inplace=True)
    else:
        gdf_vk = gdf_vk.to_crs(epsg=4326)

    # ---------------------------------------------------------
    # 4. Merge
    # ---------------------------------------------------------
    print("4. Merging datasets...")
    
    target_columns = ['id', 'user_id', 'datetaken', 'latitude', 'longitude', 'layer', 'geometry']
    
    merged_gdf = pd.concat([
        gdf_flic[target_columns], 
        gdf_twitt[target_columns], 
        gdf_vk[target_columns]
    ], ignore_index=True)
    
    merged_gdf = merged_gdf.reset_index(drop=True)
    merged_gdf.index.name = 'fid' 

    print(f"   Total merged rows: {len(merged_gdf)}")
    
    # ---------------------------------------------------------
    # 5. Export
    # ---------------------------------------------------------
    
    # A. Export EPSG:4326 
    print(f"5. Exporting to EPSG:4326 -> {out_4326}")
    merged_gdf.to_file(out_4326, driver="GPKG")
    
    # B. Export EPSG:3857 
    print(f"6. Reprojecting and exporting to EPSG:3857 -> {out_3857}")
    merged_3857 = merged_gdf.to_crs(epsg=3857)
    merged_3857.to_file(out_3857, driver="GPKG")
    
    print("Done!")

if __name__ == "__main__":
    clean_and_merge()

1. Loading and standardizing 'flic' data...
2. Loading and standardizing 'twitt' data...
3. Loading and cleaning 'vk_east' data...


C:\Users\karasoo1\AppData\Roaming\Python\Python311\site-packages\pyogrio\raw.py:198: RuntimeWarning: One or several characters couldn't be converted correctly from cp1251 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(


   Parsing VK dates...
4. Merging datasets...
   Total merged rows: 1052929
5. Exporting to EPSG:4326 -> C:\Users\karasoo1\Downloads\social_merged_4326.gpkg
6. Reprojecting and exporting to EPSG:3857 -> C:\Users\karasoo1\Downloads\social_merged_3857.gpkg
Done!


In [19]:
# ORDLO/NON-ORDLO COMPARISON IN SOCIAL MEDIA PRESENCE FOR SUPPLEMENTARY MATERIALS

# Input files
social_gpkg = os.path.join(work_dir, "social_merged_4326.gpkg")
ordlo_gpkg = os.path.join(work_dir, "ordlo.gpkg")

# Layers to analyse (Internal Name -> Display Name)
layer_config = {
    'flic': 'Flickr',
    'twitt': 'Twitter',
    'vk': 'VK'
}

def analyze_layer_stats():
    print("1. Loading datasets...")
    # Read the main social data
    gdf_social = gpd.read_file(social_gpkg)
    
    # Read the ORDLO (Occupied Territories) polygon
    gdf_ordlo = gpd.read_file(ordlo_gpkg)

    print("2. Processing dates...")
    # Ensure datetaken is a proper datetime object
    gdf_social['date'] = pd.to_datetime(gdf_social['datetaken'], errors='coerce')
    # Extract Year for grouping
    gdf_social['year'] = gdf_social['date'].dt.year

    # ---------------------------------------------------------
    # STEP 1: Spatial Join (Perform once for all data)
    # ---------------------------------------------------------
    print("3. Performing spatial join (ORDLO tagging)...")
    
    # Ensure CRS alignment (reproject ORDLO to match social data if needed)
    if gdf_social.crs != gdf_ordlo.crs:
        gdf_ordlo = gdf_ordlo.to_crs(gdf_social.crs)

    # Keep only geometry from ORDLO to avoid column name collisions
    ordlo_geom = gdf_ordlo[['geometry']].copy()

    # Join: Check if points are WITHIN ordlo polygons
    # 'predicate=within' checks if the point is inside the polygon
    joined = gpd.sjoin(gdf_social, ordlo_geom, how='left', predicate='within')

    # Create 'ordlo' column ('yes' if match found/inside, 'no' otherwise)
    joined['ordlo'] = joined['index_right'].notna().map({True: 'yes', False: 'no'})
    
    # ---------------------------------------------------------
    # STEP 2: Loop Analysis for Each Layer
    # ---------------------------------------------------------
    
    # Iterate through ['flic', 'twitt', 'vk']
    for layer_code, display_name in layer_config.items():
        print(f"\n--- Analyzing Layer: {display_name} ({layer_code}) ---")
        
        # Filter the dataset for the specific layer
        df_layer = joined[joined['layer'] == layer_code].copy()
        
        if len(df_layer) == 0:
            print(f"   Skipping {display_name}: No data found in file.")
            continue

        # Aggregate statistics: Group by Year and ORDLO status
        stats = df_layer.groupby(['year', 'ordlo']).agg(
            total_rows=('user_id', 'size'),        # Count total posts
            unique_users=('user_id', 'nunique')    # Count unique users
        ).reset_index()

        # Clean up: Drop rows with no year and ensure year is integer
        stats = stats.dropna(subset=['year'])
        stats['year'] = stats['year'].astype(int)
        
        # Sort by year for cleaner plotting
        stats = stats.sort_values('year')

        print(f"   Stats generated. Rows: {len(stats)}")

        # ---------------------------------------------------------
        # STEP 3: Plotting
        # ---------------------------------------------------------
        output_pdf = os.path.join(work_dir, f"ordlo_stats_{layer_code}.pdf")
        print(f"   Generating plot -> {output_pdf}")
        
        # Set Seaborn style
        sns.set_theme(style="whitegrid")
        
        # Create figure with 2 subplots
        fig, axes = plt.subplots(2, 1, figsize=(12, 12))

        # --- Plot A: Total Content ---
        sns.barplot(
            data=stats, 
            x='year', 
            y='total_rows', 
            hue='ordlo', 
            palette={'yes': '#d62728', 'no': '#1f77b4'}, # Red/Blue palette
            ax=axes[0]
        )
        axes[0].set_title(f'Total {display_name} content per year', fontsize=14)
        axes[0].set_ylabel('Number of posts')
        axes[0].set_xlabel('')
        axes[0].legend(title='Inside ORDLO')

        # --- Plot B: Unique Users ---
        sns.barplot(
            data=stats, 
            x='year', 
            y='unique_users', 
            hue='ordlo', 
            palette={'yes': '#d62728', 'no': '#1f77b4'},
            ax=axes[1]
        )
        axes[1].set_title(f'Unique {display_name} users per year', fontsize=14)
        axes[1].set_ylabel('Number of unique users')
        axes[1].set_xlabel('Year')
        axes[1].legend(title='Inside ORDLO')

        plt.tight_layout()
        
        # Save to PDF
        plt.savefig(output_pdf, format="pdf", bbox_inches="tight")
        plt.close(fig) # Close the figure to free memory
        
    print("\nAll processing complete!")

if __name__ == "__main__":
    analyze_layer_stats()

1. Loading datasets...
2. Processing dates...
3. Performing spatial join (ORDLO tagging)...

--- Analyzing Layer: Flickr (flic) ---
   Stats generated. Rows: 38
   Generating plot -> C:\Users\karasoo1\Downloads\ordlo_stats_flic.pdf

--- Analyzing Layer: Twitter (twitt) ---
   Stats generated. Rows: 8
   Generating plot -> C:\Users\karasoo1\Downloads\ordlo_stats_twitt.pdf

--- Analyzing Layer: VK (vk) ---
   Stats generated. Rows: 39
   Generating plot -> C:\Users\karasoo1\Downloads\ordlo_stats_vk.pdf

All processing complete!


In [5]:
# FILTERING PRESENCES BY NDVI AND LAND COVER

# Inputs
input_gpkg = os.path.join(work_dir, "social_merged_4326.gpkg")
esa_raster = os.path.join(work_dir, "ESA_WorldCover_2021_Ukraine_AOI.tif")
ndvi_raster = os.path.join(work_dir, "Summer_NDVI_Median_2019_2021.tif")

# Outputs
out_4326 = os.path.join(work_dir, "social_filtered_nature_4326.gpkg")
out_3857 = os.path.join(work_dir, "social_filtered_nature_3857.gpkg")

# Filter Criteria
# Keeping: Trees(10), Shrub(20), Grass(30), Crop(40), Bare(60), Water(80), Wetland(90), Moss(100)
# Excluding: Built-up(50), Snow(70), Mangroves(95)
target_esa_classes = [10, 20, 30, 40, 60, 80, 90, 100] 
min_ndvi = 0.5

def sample_raster_values(gdf, raster_path, column_name):
    """Samples raster values at the point locations in the GeoDataFrame."""
    print(f"   Sampling {column_name} from: {os.path.basename(raster_path)}...")
    
    # create a list of (lon, lat) coordinates
    coord_list = [(x, y) for x, y in zip(gdf.geometry.x, gdf.geometry.y)]

    # Open raster and sample
    with rasterio.open(raster_path) as src:
        # Check CRS compatibility
        if src.crs.to_epsg() != 4326:
            print(f"   WARNING: Raster CRS is {src.crs}, but points are 4326. Results may be wrong if not aligned.")
        
        # sample() returns a generator of arrays (one value per band)
        # We take the first value [0] since these are single-band rasters
        sampled_values = [x[0] for x in src.sample(coord_list)]

    gdf[column_name] = sampled_values
    return gdf

def process_spatial_attributes():
    print("1. Loading point data...")
    gdf = gpd.read_file(input_gpkg)
    print(f"   Total initial points: {len(gdf)}")

    # ---------------------------------------------------------
    # STEP 1: Sample Raster Values
    # ---------------------------------------------------------
    print("2. Sampling raster data...")
    
    # Sample ESA WorldCover
    gdf = sample_raster_values(gdf, esa_raster, 'ESA')
    
    # Sample NDVI
    gdf = sample_raster_values(gdf, ndvi_raster, 'NDVI')

    # ---------------------------------------------------------
    # STEP 2: Apply Filters
    # ---------------------------------------------------------
    print("\n3. Applying filters...")
    
    # A. Filter by ESA Class (Keep only natural/semi-natural classes)
    mask_esa = gdf['ESA'].isin(target_esa_classes)
    
    # B. Filter by NDVI (Green vegetation check)
    mask_ndvi = gdf['NDVI'] >= min_ndvi
    
    # Combine masks
    final_gdf = gdf.loc[mask_esa & mask_ndvi].copy()
    
    print("-" * 30)
    print(f"   Points removed by ESA filter (Built-up/etc): {len(gdf) - mask_esa.sum()}")
    print(f"   Points removed by NDVI < {min_ndvi}:          {len(gdf) - mask_ndvi.sum()}")
    print(f"   --------------------------------------------")
    print(f"   Final Remaining Points: {len(final_gdf)}")
    print("-" * 30)

    if len(final_gdf) == 0:
        print("   No data left after filtering. Check CRS alignment or filter strictness.")
        return

    # ---------------------------------------------------------
    # STEP 3: Export
    # ---------------------------------------------------------
    
    # A. Export EPSG:4326 (Lat/Lon)
    print(f"\n4. Exporting EPSG:4326 -> {os.path.basename(out_4326)}")
    final_gdf.to_file(out_4326, driver="GPKG")

    # B. Export EPSG:3857 (Web Mercator)
    print(f"5. Reprojecting and exporting EPSG:3857 -> {os.path.basename(out_3857)}")
    final_gdf_3857 = final_gdf.to_crs(epsg=3857)
    final_gdf_3857.to_file(out_3857, driver="GPKG")
    
    print("Done!")

if __name__ == "__main__":
    process_spatial_attributes()

1. Loading point data...
   Total initial points: 1052929
2. Sampling raster data...
   Sampling ESA from: ESA_WorldCover_2021_Ukraine_AOI.tif...
   Sampling NDVI from: Summer_NDVI_Median_2019_2021.tif...

3. Applying filters...
------------------------------
   Points removed by ESA filter (Built-up/etc): 480281
   Points removed by NDVI < 0.5:          537892
   --------------------------------------------
   Final Remaining Points: 401507
------------------------------

4. Exporting EPSG:4326 -> social_filtered_nature_4326.gpkg
5. Reprojecting and exporting EPSG:3857 -> social_filtered_nature_3857.gpkg
Done!


In [5]:
# FILTERING ROADS BY STATUS
# Input files
input_gpkg = os.path.join(work_dir, "roads_east3857.gpkg")
output_gpkg = os.path.join(work_dir, "roads_main_3857.gpkg")
layer_name = "output"

# Definition of "Main" roads (Motorway down to Tertiary)
main_categories = [
    'motorway', 'motorway_link',
    'trunk', 'trunk_link',
    'primary', 'primary_link',
    'secondary', 'secondary_link',
    'tertiary', 'tertiary_link'
]

def filter_main_roads():
    print(f"1. Loading roads from {os.path.basename(input_gpkg)}...")
    gdf = gpd.read_file(input_gpkg, layer=layer_name)
    
    print(f"   Total initial roads: {len(gdf)}")

    # ---------------------------------------------------------
    # Filter
    # ---------------------------------------------------------
    print("2. Filtering for main road categories...")
    
    # Check if 'fclass' exists
    if 'fclass' not in gdf.columns:
        print("Error: 'fclass' column not found.")
        return

    # Filter rows
    mask = gdf['fclass'].isin(main_categories)
    gdf_main = gdf.loc[mask].copy()
    
    print(f"   Roads kept (Main network): {len(gdf_main)}")
    print(f"   Categories included: {sorted(gdf_main['fclass'].unique())}")

    # ---------------------------------------------------------
    # Export
    # ---------------------------------------------------------
    if len(gdf_main) > 0:
        print(f"3. Exporting to {os.path.basename(output_gpkg)}...")
        gdf_main.to_file(output_gpkg, driver="GPKG")
        print("Done!")
    else:
        print("No roads matched the filter.")

if __name__ == "__main__":
    filter_main_roads()

1. Loading roads from roads_east3857.gpkg...
   Total initial roads: 385099
2. Filtering for main road categories...
   Roads kept (Main network): 20249
   Categories included: ['motorway', 'motorway_link', 'primary', 'primary_link', 'secondary', 'secondary_link', 'tertiary', 'tertiary_link', 'trunk', 'trunk_link']
3. Exporting to roads_main_3857.gpkg...
Done!


In [6]:
# FILTERING PRESENCES BY EXCLUSION ZONES (BUILDINGS, ROADS) AND TEMPORAL SCOPE OF THE STUDY
# Main Input (Nature-filtered data in 3857)
input_gpkg = os.path.join(work_dir, "social_filtered_nature_3857.gpkg")

# Exclusion Layers (Railways, Roads, and Buildings)
exclusion_files = [
    os.path.join(work_dir, "railways_buffer.gpkg"),
    os.path.join(work_dir, "roads_buffer_3857.gpkg"),
    os.path.join(work_dir, "buildings_overture_buffer.gpkg"),
    os.path.join(work_dir, "buildings_east_10m_buffer.gpkg")
]

# Outputs
out_3857 = os.path.join(work_dir, "social_final_filtered_3857.gpkg")
out_4326 = os.path.join(work_dir, "social_final_filtered_4326.gpkg")

# Filter Parameters
summer_months = [5, 6, 7, 8, 9] # May to September
start_year = 2015
end_year = 2021

def process_final_filtering():
    print("1. Loading social data...")
    try:
        gdf = gpd.read_file(input_gpkg)
    except Exception as e:
        print(f"   Error loading input file: {e}")
        return
        
    print(f"   Initial points: {len(gdf)}")

    # ---------------------------------------------------------
    # STEP 1: Temporal Filter (Season + Year Range)
    # ---------------------------------------------------------
    print(f"2. Applying temporal filter (May-Sept, {start_year}-{end_year})...")
    
    # Identify date column
    date_col = 'datetaken' if 'datetaken' in gdf.columns else 'date'
    if date_col not in gdf.columns:
        print("   Error: Could not find 'datetaken' or 'date' column.")
        return

    # Convert to datetime
    gdf['temp_date'] = pd.to_datetime(gdf[date_col], errors='coerce')
    
    # Filter 1: Season (May-Sept)
    mask_season = gdf['temp_date'].dt.month.isin(summer_months)
    
    # Filter 2: Year Range (2015-2021)
    mask_year = (gdf['temp_date'].dt.year >= start_year) & (gdf['temp_date'].dt.year <= end_year)
    
    # Combine filters
    gdf_filtered = gdf.loc[mask_season & mask_year].copy()
    
    # Cleanup temp column
    gdf_filtered = gdf_filtered.drop(columns=['temp_date'])
    
    print(f"   Points after temporal filters: {len(gdf_filtered)}")

    if len(gdf_filtered) == 0:
        print("   No data left after temporal filters. Exiting.")
        return

    # ---------------------------------------------------------
    # STEP 2: Spatial Exclusion (Rail + Road + Buildings)
    # ---------------------------------------------------------
    print("3. Building exclusion mask (Railways + Roads + Buildings)...")
    
    exclusion_geoms = []
    target_crs = gdf_filtered.crs  # Should be 3857

    for fpath in exclusion_files:
        filename = os.path.basename(fpath)
        if not os.path.exists(fpath):
            print(f"   Warning: File not found, skipping: {filename}")
            continue
            
        print(f"   Loading: {filename}...")
        try:
            # Load file (geometry only is faster for exclusions)
            gdf_ex = gpd.read_file(fpath)
            
            # Reproject if necessary
            if gdf_ex.crs != target_crs:
                print(f"      Reprojecting {filename}...")
                gdf_ex = gdf_ex.to_crs(target_crs)
            
            exclusion_geoms.append(gdf_ex[['geometry']])
            
        except Exception as e:
            print(f"      Error reading {filename}: {e}")

    if not exclusion_geoms:
        print("   Error: No exclusion layers could be loaded.")
        return

    # Combine all exclusions into one dataframe
    print("   Combining exclusion layers...")
    full_exclusion = pd.concat(exclusion_geoms, ignore_index=True)
    full_exclusion_gdf = gpd.GeoDataFrame(full_exclusion, geometry='geometry', crs=target_crs)

    print("4. Removing points inside exclusion zones...")
    
    # SPATIAL JOIN (Difference)
    # Left Join: Keep all points, identify matches
    joined = gpd.sjoin(gdf_filtered, full_exclusion_gdf, how='left', predicate='intersects')
    
    # Filter: Keep rows where index_right is NaN (meaning NO intersection found)
    mask_safe = joined['index_right'].isna()
    gdf_final = joined.loc[mask_safe].copy()
    
    # Drop the artifact column from sjoin
    gdf_final = gdf_final.drop(columns=['index_right'])

    removed_count = len(gdf_filtered) - len(gdf_final)
    print(f"   Points removed (inside exclusion zones): {removed_count}")
    print(f"   Final remaining points: {len(gdf_final)}")

    # ---------------------------------------------------------
    # STEP 3: Export
    # ---------------------------------------------------------
    if len(gdf_final) > 0:
        # A. Export EPSG:3857
        print(f"5. Exporting EPSG:3857 -> {os.path.basename(out_3857)}")
        gdf_final.to_file(out_3857, driver="GPKG")

        # B. Export EPSG:4326
        print(f"6. Reprojecting and exporting EPSG:4326 -> {os.path.basename(out_4326)}")
        gdf_4326 = gdf_final.to_crs(epsg=4326)
        gdf_4326.to_file(out_4326, driver="GPKG")
        
        print("All Done!")
    else:
        print("No data remaining to export.")

if __name__ == "__main__":
    process_final_filtering()

1. Loading social data...
   Initial points: 401507
2. Applying temporal filter (May-Sept, 2015-2021)...
   Points after temporal filters: 141416
3. Building exclusion mask (Railways + Roads + Buildings)...
   Loading: railways_buffer.gpkg...
   Loading: roads_buffer_3857.gpkg...
   Loading: buildings_overture_buffer.gpkg...
   Loading: buildings_east_10m_buffer.gpkg...
   Combining exclusion layers...
4. Removing points inside exclusion zones...
   Points removed (inside exclusion zones): 15027
   Final remaining points: 126389
5. Exporting EPSG:3857 -> social_final_filtered_3857.gpkg
6. Reprojecting and exporting EPSG:4326 -> social_final_filtered_4326.gpkg
All Done!


In [7]:
# SPATIAL THINNING OF THE PRESENCES BY H3 CELLS

# Inputs
input_social = os.path.join(work_dir, "social_final_filtered_3857.gpkg")
input_h3 = os.path.join(work_dir, "h3_8.gpkg")

# Outputs
out_3857 = os.path.join(work_dir, "1social_h3_max_ndvi_3857.gpkg")
out_4326 = os.path.join(work_dir, "1social_h3_max_ndvi_4326.gpkg")

def subset_highest_ndvi_per_cell():
    print("1. Loading datasets...")
    try:
        gdf_social = gpd.read_file(input_social)
        gdf_h3 = gpd.read_file(input_h3)
    except Exception as e:
        print(f"   Error loading files: {e}")
        return

    print(f"   Social points: {len(gdf_social)}")
    print(f"   H3 cells: {len(gdf_h3)}")

    # ---------------------------------------------------------
    # STEP 1: Spatial Join (Assign Points to H3 Cells)
    # ---------------------------------------------------------
    print("2. Assigning points to H3 cells...")

    # Ensure CRS match
    if gdf_social.crs != gdf_h3.crs:
        print(f"   Reprojecting H3 layer to {gdf_social.crs}...")
        gdf_h3 = gdf_h3.to_crs(gdf_social.crs)
    
    # Check for common H3 column names
    h3_col = None
    possible_cols = ['h3_index', 'h3_id', 'index', 'id']
    for col in possible_cols:
        if col in gdf_h3.columns:
            h3_col = col
            break
            
    # If no named column found, create one from the index
    if h3_col is None:
        print("   No standard H3 index column found. Using row index as identifier.")
        gdf_h3['h3_cell_id'] = gdf_h3.index
        h3_col = 'h3_cell_id'

    # Keep only geometry and the ID column for the join to save memory
    h3_subset = gdf_h3[[h3_col, 'geometry']].copy()

    # Spatial Join: Points WITHIN Polygons
    joined = gpd.sjoin(gdf_social, h3_subset, how='inner', predicate='within')
    
    print(f"   Points successfully assigned to a cell: {len(joined)}")

    # ---------------------------------------------------------
    # STEP 2: Filter (Max NDVI per Cell)
    # ---------------------------------------------------------
    print("3. Selecting point with highest NDVI per cell...")

    if 'NDVI' not in joined.columns:
        print("   Error: 'NDVI' column not found in social data.")
        return

    # Sort by H3 ID and then NDVI (Descending)
    # This puts the highest NDVI for each cell at the top of its group
    joined_sorted = joined.sort_values(by=[h3_col, 'NDVI'], ascending=[True, False])

    # Drop duplicates keeping the FIRST occurrence (which is the Max NDVI)
    gdf_final = joined_sorted.drop_duplicates(subset=[h3_col], keep='first').copy()

    # Clean up the join columns (h3 index and index_right)
    if 'index_right' in gdf_final.columns:
        gdf_final = gdf_final.drop(columns=['index_right'])

    print(f"   Final count (Unique cells with data): {len(gdf_final)}")

    # ---------------------------------------------------------
    # STEP 3: Export
    # ---------------------------------------------------------
    if len(gdf_final) > 0:
        # A. Export EPSG:3857
        print(f"4. Exporting EPSG:3857 -> {os.path.basename(out_3857)}")
        gdf_final.to_file(out_3857, driver="GPKG")

        # B. Export EPSG:4326
        print(f"5. Reprojecting and exporting EPSG:4326 -> {os.path.basename(out_4326)}")
        gdf_4326 = gdf_final.to_crs(epsg=4326)
        gdf_4326.to_file(out_4326, driver="GPKG")
        
        print("Done!")
    else:
        print("No data remaining.")

if __name__ == "__main__":
    subset_highest_ndvi_per_cell()

1. Loading datasets...
   Social points: 126389
   H3 cells: 121383
2. Assigning points to H3 cells...
   Points successfully assigned to a cell: 126056
3. Selecting point with highest NDVI per cell...
   Final count (Unique cells with data): 20124
4. Exporting EPSG:3857 -> 1social_h3_max_ndvi_3857.gpkg
5. Reprojecting and exporting EPSG:4326 -> 1social_h3_max_ndvi_4326.gpkg
Done!


In [1]:
# GENERATING PSEUDO-ABSENCES (20K)

# Inputs
input_h3 = os.path.join(work_dir, "h3_8.gpkg")
esa_raster = os.path.join(work_dir, "ESA_WorldCover_2021_Ukraine_AOI.tif")

# Exclusion Layers
exclusion_files = [
    os.path.join(work_dir, "railways_buffer.gpkg"),
    os.path.join(work_dir, "roads_buffer_3857.gpkg"), 
    os.path.join(work_dir, "buildings_overture_buffer.gpkg"),
    os.path.join(work_dir, "buildings_east_10m_buffer.gpkg")
]

# Outputs
out_3857 = os.path.join(work_dir, "random_locations_20k_3857.gpkg")
out_4326 = os.path.join(work_dir, "random_locations_20k_4326.gpkg")

# Parameters
min_dist = 50.0  # meters
initial_n = 35000 # Generate extra to account for ESA filtering loss
final_n = 20000   # Target output
target_esa_classes = [10, 20, 30, 40, 60, 80, 90, 100]

def generate_random_points_optimized():
    print("1. Loading H3 Grid...")
    gdf_h3 = gpd.read_file(input_h3)
    target_crs = gdf_h3.crs # Should be 3857
    print(f"   Total H3 cells: {len(gdf_h3)}")

    # ---------------------------------------------------------
    # STEP 1: Exclusion Masking (Cells + Neighbors)
    # ---------------------------------------------------------
    print("2. Building exclusion zone...")
    exclusion_geoms = []
    
    for fpath in exclusion_files:
        if os.path.exists(fpath):
            try:
                # Read geometry only for speed
                gdf_ex = gpd.read_file(fpath)
                if gdf_ex.crs != target_crs:
                    gdf_ex = gdf_ex.to_crs(target_crs)
                exclusion_geoms.append(gdf_ex[['geometry']])
            except Exception as e:
                print(f"   Warning: Could not load {os.path.basename(fpath)}: {e}")
    
    if not exclusion_geoms:
        print("Error: No exclusion files loaded.")
        return

    # Combine exclusions
    gdf_exclusions = pd.concat(exclusion_geoms, ignore_index=True)
    gdf_exclusions = gpd.GeoDataFrame(gdf_exclusions, geometry='geometry', crs=target_crs)

    print("   Identifying 'bad' cells (intersecting exclusions)...")
    # Identify cells that overlap with exclusion zones
    bad_cells_join = gpd.sjoin(gdf_h3, gdf_exclusions, how='inner', predicate='intersects')
    bad_indices = bad_cells_join.index.unique()
    
    print(f"   Directly intersecting cells: {len(bad_indices)}")
    
    # Identify Neighbors:
    print("   Identifying neighbors of bad cells...")
    gdf_bad = gdf_h3.loc[bad_indices]
    
    # Find all cells that touch the bad cells
    neighbor_join = gpd.sjoin(gdf_h3, gdf_bad, how='inner', predicate='touches')
    neighbor_indices = neighbor_join.index.unique()
    
    # Combine lists to remove
    ids_to_remove = set(bad_indices).union(set(neighbor_indices))
    
    # Create Safe Mask
    gdf_safe = gdf_h3.drop(index=list(ids_to_remove)).copy()
    print(f"   Safe cells remaining: {len(gdf_safe)}")

    if len(gdf_safe) == 0:
        print("Error: No safe cells remaining.")
        return

    # ---------------------------------------------------------
    # STEP 2: Generate Random Points (Optimized)
    # ---------------------------------------------------------
    print(f"3. Selecting {initial_n} random SAFE cells...")
    
    # Check if we have enough cells
    if len(gdf_safe) < initial_n:
        print(f"   Warning: Only {len(gdf_safe)} safe cells available. Using all of them.")
        gdf_subset = gdf_safe
    else:
        # Randomly select 35K cells
        gdf_subset = gdf_safe.sample(n=initial_n)

    print("   Generating 1 random point per selected cell...")
    
    # Use sample_points(1) which generates 1 point per geometry
    # This returns a GeoSeries of points
    points_gs = gdf_subset.sample_points(size=1).explode(index_parts=False).reset_index(drop=True)
    
    print(f"   Points generated: {len(points_gs)}")

    # ---------------------------------------------------------
    # STEP 3: Enforce Minimum Distance (50m)
    # ---------------------------------------------------------
    print("4. Enforcing 50m minimum distance...")
    
    # Extract coordinates
    coords = np.array([(p.x, p.y) for p in points_gs])
    
    # Build KDTree
    tree = cKDTree(coords)
    
    # Find pairs within min_dist
    pairs = tree.query_pairs(min_dist)
    
    # Greedy removal
    drop_indices = set()
    for i, j in pairs:
        if i not in drop_indices and j not in drop_indices:
            drop_indices.add(j)
            
    # Filter points
    keep_indices = [i for i in range(len(coords)) if i not in drop_indices]
    
    # Create GeoDataFrame from kept points
    final_coords = coords[keep_indices]
    gdf_points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(final_coords[:,0], final_coords[:,1]),
        crs=target_crs
    )
    
    print(f"   Points remaining after distance check: {len(gdf_points)}")

    # ---------------------------------------------------------
    # STEP 4: Sample ESA Raster
    # ---------------------------------------------------------
    print("5. Sampling ESA Land Cover...")
    
    coord_list = [(x, y) for x, y in zip(gdf_points.geometry.x, gdf_points.geometry.y)]
    
    sampled_values = []
    
    with rasterio.open(esa_raster) as src:
        # Handle CRS mismatch if raster is not 3857
        if src.crs.to_epsg() != 3857:
            # Reproject points momentarily for sampling
            gdf_sample_proj = gdf_points.to_crs(src.crs)
            coord_list_proj = [(x, y) for x, y in zip(gdf_sample_proj.geometry.x, gdf_sample_proj.geometry.y)]
            sampled_values = [x[0] for x in src.sample(coord_list_proj)]
        else:
            sampled_values = [x[0] for x in src.sample(coord_list)]

    gdf_points['ESA'] = sampled_values

    # ---------------------------------------------------------
    # STEP 5: Filter & Final Reduction
    # ---------------------------------------------------------
    print("6. Filtering by ESA class and reducing to target count...")
    
    # Filter classes
    mask_esa = gdf_points['ESA'].isin(target_esa_classes)
    gdf_valid = gdf_points.loc[mask_esa].copy()
    
    print(f"   Points matching ESA criteria: {len(gdf_valid)}")
    
    # Reduce to 20 K
    if len(gdf_valid) > final_n:
        print(f"   Downsampling from {len(gdf_valid)} to {final_n}...")
        gdf_final = gdf_valid.sample(n=final_n, random_state=42)
    else:
        print(f"   Warning: Only found {len(gdf_valid)} valid points (Target: {final_n}). Returning all.")
        gdf_final = gdf_valid

    # ---------------------------------------------------------
    # STEP 6: Export
    # ---------------------------------------------------------
    print(f"7. Exporting {len(gdf_final)} points...")
    
    # 3857
    print(f"   Saving -> {os.path.basename(out_3857)}")
    gdf_final.to_file(out_3857, driver="GPKG")
    
    # 4326
    print(f"   Saving -> {os.path.basename(out_4326)}")
    gdf_final_4326 = gdf_final.to_crs(epsg=4326)
    gdf_final_4326.to_file(out_4326, driver="GPKG")
    
    print("Done!")

if __name__ == "__main__":
    generate_random_points_optimized()

1. Loading H3 Grid...
   Total H3 cells: 121383
2. Building exclusion zone...
   Identifying 'bad' cells (intersecting exclusions)...
   Directly intersecting cells: 40275
   Identifying neighbors of bad cells...
   Safe cells remaining: 43644
3. Selecting 35000 random SAFE cells...
   Generating 1 random point per selected cell...
   Points generated: 35000
4. Enforcing 50m minimum distance...
   Points remaining after distance check: 34998
5. Sampling ESA Land Cover...
6. Filtering by ESA class and reducing to target count...
   Points matching ESA criteria: 34163
   Downsampling from 34163 to 20000...
7. Exporting 20000 points...
   Saving -> random_locations_20k_3857.gpkg
   Saving -> random_locations_20k_4326.gpkg
Done!
